# AI texts from open models (issue #46, work package 10)

Runs `training/dataset/generate.py run` for one open model of `generation.json` in a GPU session on Colab (a T4 is enough: 7-8B models are loaded in 4 bit). The API models run locally (`generate.py run <name>`), no GPU needed.

Before the first run:
- **Prompts.** `generate.py prompts` locally, then copy `training/data/generation/prompts.jsonl` into a folder of your Google Drive. It contains human texts: do not share the folder, delete it when the generation is done.
- **Gated models** (Llama, Mistral): accept the licence on the model page on Hugging Face, and add a token as the Colab secret `HF_TOKEN` (key symbol on the left).
- **GPU.** Runtime → Change runtime type → T4 GPU.

One generator per session. The output is written to the Drive after every batch, so running the notebook again continues where it stopped. Look at the samples at the end before the next generator.

In [ ]:
GENERATOR = "qwen2.5-7b-instruct"    # an open model of generation.json (backend "transformers")
COMMIT = "own-model-training-57"      # branch or commit of the repository
PROMPTS = "/content/drive/MyDrive/aivsai-generation/prompts.jsonl"
OUT = "/content/drive/MyDrive/aivsai-generation/outputs"   # one file per generator; copy it to training/data/generation/outputs/
HOURS = 3.5                           # stop before the session ends; running again continues
BATCH = 16                            # prompts per batch; lower it if the GPU runs out of memory
LIMIT = ""                            # e.g. "--limit 32" to look at a few first

In [ ]:
import os, pathlib
from google.colab import drive, userdata

drive.mount("/content/drive")
assert pathlib.Path(PROMPTS).exists(), f"{PROMPTS} is missing: copy prompts.jsonl to the Drive"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: gated models (Llama, Mistral) will not load")

%cd /content
!rm -rf ai-check-extension && git clone -q https://github.com/OJPOJ/ai-check-extension.git && git -C ai-check-extension checkout -q $COMMIT && git -C ai-check-extension log -1 --oneline
!pip install -q "transformers>=4.56" accelerate bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content/ai-check-extension/training
!python dataset/generate.py run $GENERATOR --prompts $PROMPTS --out $OUT --hours $HOURS --batch $BATCH $LIMIT

Samples to look at: refusals, "Here is ..." in front of the text, titles or lists despite the request, texts cut off, another language. Whatever comes up often belongs in `clean_output()` of `generate.py`, applied to every generator.

In [ ]:
import json, random, collections

rows = [json.loads(line) for line in open(f"{OUT}/{GENERATOR}.jsonl", encoding="utf-8")]
print(len(rows), "texts;", collections.Counter(r["finish"] for r in rows))
for r in random.sample(rows, min(8, len(rows))):
    print("-" * 80, "\n", r["finish"], "\n", (r["text"] or "")[:700])